In [1]:
# xgboost_model
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score
import xgboost as xgb
import joblib
import os

# 1) Load dataset
df = pd.read_excel("Diagnosis3.xlsx")

# 2) Preprocessing
df['DOB'] = pd.to_datetime(df['DOB'], errors='coerce')
df['DOS From'] = pd.to_datetime(df['DOS From'], errors='coerce')
df['Age'] = ((df['DOS From'] - df['DOB']).dt.days // 365).fillna(-1).astype(int)
df['DOS_Weekday'] = df['DOS From'].dt.weekday.fillna(-1).astype(int)
df['DOS_Month'] = df['DOS From'].dt.month.fillna(-1).astype(int)
if 'Mem.#' in df.columns: df.drop(columns=['Mem.#'], inplace=True)
for c in ['DOB','DOS From']: df.drop(columns=[c], inplace=True)

target_col = 'D1'
df[target_col] = df[target_col].fillna("None")
feature_cols = [c for c in df.columns if c != target_col]

# 3) Group rare labels
rare_threshold = 20
counts = df[target_col].value_counts()
rare_labels = counts[counts < rare_threshold].index.tolist()
df['D1_grouped'] = df[target_col].apply(lambda x: 'Other' if x in rare_labels else x)
target = 'D1_grouped'

# 4) Encode features
X = df[feature_cols].copy()
cat_cols = X.select_dtypes(include=['object','category']).columns.tolist()
feature_label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str).fillna("None"))
    feature_label_encoders[col] = le
X = X.apply(pd.to_numeric, errors='coerce').fillna(-1).astype(np.int32)

# 5) Train/validation/test split
y = df[[target]]
X_trn, X_tst, y_trn, y_tst = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y[target])
X_trn, X_vld, y_trn, y_vld = train_test_split(X_trn, y_trn, test_size=0.2, random_state=42, stratify=y_trn[target])

# 6) Encode target
target_le = LabelEncoder()
y_trn_enc = target_le.fit_transform(y_trn[target])
y_vld_enc = target_le.transform(y_vld[target])
y_tst_enc = target_le.transform(y_tst[target])

# 7) Fit XGBoost
clf = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=len(np.unique(y_trn_enc)),
    tree_method='hist',
    max_depth=6,
    learning_rate=0.05,
    n_estimators=500,
    early_stopping_rounds=30
)
classes, counts = np.unique(y_trn_enc, return_counts=True)
class_weights = {c: max(counts)/count for c,count in zip(classes, counts)}
sample_weights = np.array([class_weights[c] for c in y_trn_enc], dtype=np.float32)
clf.fit(X_trn, y_trn_enc, sample_weight=sample_weights, eval_set=[(X_vld, y_vld_enc)], verbose=True)

# 8) Evaluate
y_pred = clf.predict(X_tst)
acc = accuracy_score(y_tst_enc, y_pred)
f1 = f1_score(y_tst_enc, y_pred, average='weighted')
print(f"XGBoost Test Accuracy: {acc:.4f}, Weighted F1: {f1:.4f}")

# 9) Save model
os.makedirs("models", exist_ok=True)
joblib.dump(clf, "models/xgb_model.joblib")
joblib.dump(feature_label_encoders, "models/feature_label_encoders.joblib")
joblib.dump(target_le, "models/target_le.joblib")

[0]	validation_0-mlogloss:3.58480
[1]	validation_0-mlogloss:3.36448
[2]	validation_0-mlogloss:3.21228
[3]	validation_0-mlogloss:3.09385
[4]	validation_0-mlogloss:2.99341
[5]	validation_0-mlogloss:2.90754
[6]	validation_0-mlogloss:2.83250
[7]	validation_0-mlogloss:2.76607
[8]	validation_0-mlogloss:2.70599
[9]	validation_0-mlogloss:2.64972
[10]	validation_0-mlogloss:2.59474
[11]	validation_0-mlogloss:2.54824
[12]	validation_0-mlogloss:2.50211
[13]	validation_0-mlogloss:2.46102
[14]	validation_0-mlogloss:2.42038
[15]	validation_0-mlogloss:2.38280
[16]	validation_0-mlogloss:2.34617
[17]	validation_0-mlogloss:2.31086
[18]	validation_0-mlogloss:2.27770
[19]	validation_0-mlogloss:2.24679
[20]	validation_0-mlogloss:2.21743
[21]	validation_0-mlogloss:2.18894
[22]	validation_0-mlogloss:2.16247
[23]	validation_0-mlogloss:2.13817
[24]	validation_0-mlogloss:2.11580
[25]	validation_0-mlogloss:2.09295
[26]	validation_0-mlogloss:2.07161
[27]	validation_0-mlogloss:2.05062
[28]	validation_0-mlogloss:2.0

['models/target_le.joblib']